In [1]:
import os

#os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="7"
from datasets import load_dataset, Dataset, DatasetDict
from dataclasses import dataclass, field
from typing import Optional
import torch
import torch.nn as nn
from peft import LoraConfig
from tqdm import tqdm
import pandas as pd
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, HfArgumentParser, TrainingArguments, AutoTokenizer, pipeline

from trl import SFTTrainer
import random
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling
import pickle 
tqdm.pandas()

In [2]:
def preprocessing_data(data, df):
    data_new = [list(x) for x in (data[:][0])]

    note = []
    subject_id = []

    for i in data_new:
        note.append(i[0])
        subject_id.append(i[1])

    df_train = pd.DataFrame({'note': note,'SUBJECT_ID': subject_id})

    df_train_merged = df_train.merge(df[['SUBJECT_ID', 'code_name']], on='SUBJECT_ID', how='left')

    return df_train_merged

import ast

def processing_df(df_data):
    # 데이터 로드하기
    #icd9_descript = pd.read_csv('physionet.org/files/clinical-bert-mimic-notes/1.0.0/setup_outputs/ICD9_Descriptions.csv')
    medcat_descript = pd.read_csv('dataset/MIMIC/MedCAT_Descriptions.csv')
    reidentified_subject_ids = pd.read_csv('dataset/MIMIC/reidentified_subject_ids.csv')
    #subject_id_to_icd9 = pd.read_csv('physionet.org/files/clinical-bert-mimic-notes/1.0.0/setup_outputs/SUBJECT_ID_to_ICD9.csv')
    subject_id_to_medcat = pd.read_csv('dataset/MIMIC/SUBJECT_ID_to_MedCAT.csv')
    subject_id_to_name = pd.read_csv('dataset/MIMIC/SUBJECT_ID_to_NAME.csv')

    # SUBJECT_ID별 해당되는 code_name 매칭하기
    medcat_name_subject = pd.merge(subject_id_to_medcat, medcat_descript, on='CODE', how='inner')
    subject_medcat_names = medcat_name_subject.groupby(['SUBJECT_ID'])['DESCRIPTION'].apply(list).reset_index(name='code_name')

    df_data['code_name']= df_data['code_name'].apply(lambda x: (ast.literal_eval(x)))
    df_data['condition_nums'] = df_data['code_name'].apply(lambda x: len(x))

    subject_id_to_name['FULL_NAME'] = subject_id_to_name['FIRST_NAME'] + ' ' + subject_id_to_name['LAST_NAME']
    fin_df = pd.merge(subject_id_to_name[['SUBJECT_ID', 'FULL_NAME']], df_data, on='SUBJECT_ID', how='inner')

    fin_df = fin_df[['SUBJECT_ID', 'FULL_NAME', 'condition_nums', 'note', 'code_name']]
    fin_df.columns = ['SUBJECT_ID', 'name', 'num', 'note', 'condition']

    fin_df['condition'] = fin_df['condition'].apply(lambda x: ", ".join(x))

    return fin_df

In [3]:
from datasets import load_dataset, Dataset, load_from_disk
import pandas as pd

with open("dataset/MIMIC/train_data.pickle", "rb") as f:
    train_data = pickle.load(f)

with open("dataset/MIMIC/test_data.pickle", "rb") as f:
    test_data = pickle.load(f)

df = pd.read_csv('dataset/MIMIC/pesudo_mimic3_processed.csv')

train_df = preprocessing_data(train_data, df)
df_train = processing_df(train_df)



In [21]:
all_string = ''
for i in df_train['condition']:
    all_string = all_string +', '+ i 
all_string = all_string[2:]


In [50]:
from collections import Counter
combined_condition = all_string.split(', ')
all_condition = Counter(combined_condition)

condition = list(all_condition.keys())
count = list(all_condition.values())

df_count = pd.DataFrame({'condition': condition, 'count': count})

In [58]:
df_count['condition'] = df_count['condition'].apply(lambda x: x.lower())
df_count = df_count.sort_values(by='count', ascending=False)
df_count

,condition,count
3,edema,29935
13,dyspnea,21636
49,pain,21166
37,hypertensive disease,21128
1,coughing,20372
...,...,...
2460,vitreous degeneration,1
2602,aggressive passive personality,1
2463,multiple valve disease,1
2464,myotonic disorder,1


In [65]:
total = df_count['count'].sum()
df_count['ratio'] = df_count['count']/total*100
df_count

,condition,count,ratio
3,edema,29935,2.540928
13,dyspnea,21636,1.836496
49,pain,21166,1.796602
37,hypertensive disease,21128,1.793376
1,coughing,20372,1.729206
...,...,...,...
2460,vitreous degeneration,1,0.000085
2602,aggressive passive personality,1,0.000085
2463,multiple valve disease,1,0.000085
2464,myotonic disorder,1,0.000085


In [62]:
df_train['condition'] = df_train['condition'].apply(lambda x: x.split(", "))
df_count.describe()

,count
count,2623.000000
mean,449.147160
std,1813.684355
min,1.000000
25%,3.000000
50%,13.000000
75%,105.000000
max,29935.000000


In [148]:
print(df_count[:53]['ratio'].sum())
general_conditions = list(df_count[:53]['condition'])

50.28906395226944


In [149]:
(general_conditions)

['edema',
 'dyspnea',
 'pain',
 'hypertensive disease',
 'coughing',
 'fever',
 'chest pain',
 'wheezing',
 'essential hypertension',
 'nausea',
 'exanthema',
 'anxiety',
 'constipation',
 'vomiting',
 'osteochondritis dissecans',
 'premature ventricular contractions',
 'pleural effusion disorder',
 'abdominal pain',
 'pneumonia',
 'aortic valve insufficiency',
 'weakness',
 'lethargy',
 'anemia',
 'cyanosis',
 'confusion',
 'headache',
 'flushing',
 'congestive heart failure',
 'erythema',
 'myocardial infarction',
 'hematuria',
 'cerebrovascular accident',
 'deep vein thrombosis',
 'obesity',
 'chill fever',
 'pneumothorax',
 'syncope',
 'flatulence',
 'diabetes mellitus',
 'aortic valve stenosis',
 'diarrhea',
 'nausea and vomiting',
 'simian aids',
 'fatigue',
 'apnea',
 'hyperlipidemia',
 'icterus',
 'diabetes',
 'pulmonary edema',
 'urinary tract infection',
 'gastroesophageal reflux disease',
 'lymphadenopathy',
 'tremor']

In [152]:
df_train['condition'] = df_train['condition'].apply(lambda x: x.split(','))
df_train

<ipython-input-152-f65254403cac>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_train['condition'] = df_train['condition'].apply(lambda x: x.split(','))


,SUBJECT_ID,name,a,condition,condition_nums,overlap
0,4,Wyman Maruca,2,"[alkaloses, Coughing, Diabetes, Edema, Wheezin...",29,0
1,5,Marino Didomenico,1,[Exanthema],1,0
2,8,Elpidio Mcelrath,3,"[Asthma, Depressive disorder, Meconium Aspirat...",4,0
3,9,Lubertha Dansie,2,"[Coughing, Edema, Congestive heart failure, Ac...",27,0
4,10,Dwaine Hedglin,3,"[Apnea, Simian AIDS, Hepatitis B, Enlarged liv...",6,0
...,...,...,...,...,...,...
40350,99985,Angela Mariner,2,"[Chest Pain, Coughing, Non-Insulin-Dependent D...",83,0
40351,99991,Braden Rostami,2,"[Chest Pain, Coughing, Diabetes, Edema, Flushi...",33,0
40352,99992,Elaine Frederick,2,"[Chest Pain, Edema, Flushing, Congestive heart...",21,0
40353,99995,Barry Heise,2,"[Diabetes, Hypertensive disease, Pain, Constip...",8,0


In [ ]:
df_train['overlap'] = df_train['condition'].apply(lambda x: sum(c.lower() in [word.lower() for word in x] for c in general_conditions))
df_train['overlap_ratio'] = df_train['overlap']/df_train['condition_nums']*100
df_train['a'] = 0
df_train['b'] = ''

In [154]:
df_train

,SUBJECT_ID,name,a,condition,condition_nums,overlap,overlap_ratio,b
0,4,Wyman Maruca,0,"[alkaloses, Coughing, Diabetes, Edema, Wheezin...",29,12,41.379310,
1,5,Marino Didomenico,0,[Exanthema],1,1,100.000000,
2,8,Elpidio Mcelrath,0,"[Asthma, Depressive disorder, Meconium Aspirat...",4,0,0.000000,
3,9,Lubertha Dansie,0,"[Coughing, Edema, Congestive heart failure, Ac...",27,16,59.259259,
4,10,Dwaine Hedglin,0,"[Apnea, Simian AIDS, Hepatitis B, Enlarged liv...",6,3,50.000000,
...,...,...,...,...,...,...,...,...
40350,99985,Angela Mariner,0,"[Chest Pain, Coughing, Non-Insulin-Dependent D...",83,32,38.554217,
40351,99991,Braden Rostami,0,"[Chest Pain, Coughing, Diabetes, Edema, Flushi...",33,22,66.666667,
40352,99992,Elaine Frederick,0,"[Chest Pain, Edema, Flushing, Congestive heart...",21,13,61.904762,
40353,99995,Barry Heise,0,"[Diabetes, Hypertensive disease, Pain, Constip...",8,5,62.500000,


In [ ]:
df_train['a'] = df_train['overlap_ratio'].apply(lambda x: 3 if x < 10 else 2 if x < 50 else 1)
df_train['b'] = df_train['a'].apply(lambda x: 'final group3' if x ==3 else 'final group2' if x != 1 else 'final group1')
df_train

In [121]:
df_train = df_train[['SUBJECT_ID', 'name', 'a', 'condition', 'num']]
df_train.columns = ['SUBJECT_ID', 'name', 'a', 'condition', 'condition_nums']
df_train

,SUBJECT_ID,name,a,condition,condition_nums
0,4,Wyman Maruca,2,"[alkaloses, Coughing, Diabetes, Edema, Wheezin...",29
1,5,Marino Didomenico,1,[Exanthema],1
2,8,Elpidio Mcelrath,3,"[Asthma, Depressive disorder, Meconium Aspirat...",4
3,9,Lubertha Dansie,2,"[Coughing, Edema, Congestive heart failure, Ac...",27
4,10,Dwaine Hedglin,3,"[Apnea, Simian AIDS, Hepatitis B, Enlarged liv...",6
...,...,...,...,...,...
40350,99985,Angela Mariner,2,"[Chest Pain, Coughing, Non-Insulin-Dependent D...",83
40351,99991,Braden Rostami,2,"[Chest Pain, Coughing, Diabetes, Edema, Flushi...",33
40352,99992,Elaine Frederick,2,"[Chest Pain, Edema, Flushing, Congestive heart...",21
40353,99995,Barry Heise,2,"[Diabetes, Hypertensive disease, Pain, Constip...",8


In [ ]:
df_train['condition'] = df_train['condition'].apply(lambda x: ', '.join(x))
df_train

In [159]:
df_train[df_train['a']==3]

,SUBJECT_ID,name,a,condition,condition_nums,overlap,overlap_ratio,b
2,8,Elpidio Mcelrath,3,"Asthma, Depressive disorder, Meconium Aspirati...",4,0,0.000000,final group3
6,16,Johna Scheve,3,click hip,1,0,0.000000,final group3
46,74,Adeline Schlager,3,"Cardiac Arrest, Diabetic Retinopathy, Rubella",3,0,0.000000,final group3
78,118,Norman Loan,3,"Migraine Disorders, HEMOLYTIC, THALASSEMIA MIN...",2,0,0.000000,final group3
80,122,Dyann Schwimmer,3,"Depressive disorder, Complete atrioventricular...",3,0,0.000000,final group3
...,...,...,...,...,...,...,...,...
34859,76646,Gretta Redlinger,3,"Deep Vein Thrombosis, Back Pain, Spasm, Famili...",11,1,9.090909,final group3
35433,79096,Ladonna Ebbing,3,"Cerebral Edema, Chronic multifocal osteomyelitis",2,0,0.000000,final group3
36854,85055,Fletcher Vanderkooi,3,"Cardiac Arrest, Blood Coagulation Disorders",2,0,0.000000,final group3
37515,88025,Wheeler Lowy,3,Chronic multifocal osteomyelitis,1,0,0.000000,final group3


In [160]:
import random

df_train_1 = df_train[df_train['a']==1].sample(n=1333, replace=False, random_state=42)[['SUBJECT_ID', 'name', 'condition', 'condition_nums', 'a']]
df_train_2 = df_train[df_train['a']==2].sample(n=1334, replace=False, random_state=42)[['SUBJECT_ID', 'name', 'condition', 'condition_nums', 'a']]
df_train_3 = df_train[df_train['a']==3].sample(n=1333, replace=False, random_state=42)[['SUBJECT_ID', 'name', 'condition', 'condition_nums', 'a']]

In [ ]:
sample_b = pd.concat([df_train_1,df_train_2,df_train_3], axis=0)
sample_b.a.value_counts()

In [162]:
#sample_b = sample_b.drop('a', axis=1)
sample_b.to_csv('sample_b.csv', index=False)